# 锚点定位

学习目标：能把说明框的位置和尺寸关联到锚点，并结合 Popover 和位置候选处理边缘溢出。

前置知识：absolute/fixed 定位、包含块、层叠上下文、条件规则、HTML 按钮和焦点操作。

适用范围：CSS Anchor Positioning Level 1 的演进中功能，使用现代 position-try-fallbacks 命名。锚点关联、尺寸函数、位置回退与 Popover 要分别核对浏览器支持；页面不使用 JavaScript。

工作目录：`content/Web与应用开发/css/`（以下命令从项目根目录切换到这里执行）。

环境准备：[环境配置与运行](README.md)。

配套脚本：位于 scripts/25-anchor-positioning/。

1. [index.html](scripts/25-anchor-positioning/index.html)：普通说明框的锚点坐标与尺寸。
2. [popover.html](scripts/25-anchor-positioning/popover.html)：原生浮层、顶层和位置回退。
3. [styles.css](scripts/25-anchor-positioning/styles.css)：锚点关联与位置候选。

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/css
```

Step 2：启动本章预览服务。

```bash
python -m http.server 8101 --bind 127.0.0.1
```

Step 3：打开[本章示例首页](http://127.0.0.1:8101/scripts/25-anchor-positioning/index.html)。

保存修改后刷新页面。

Step 4：在服务终端按 Ctrl+C 停止服务。

## 1 参照元素与定位说明框

锚点定位（anchor positioning）让一个定位元素引用另一个元素的位置与尺寸。它用于说明框、菜单或浮层的布局；不负责决定何时打开，也不会自动赋予“提示框”语义。

.anchor 是本例的参照元素，.callout 是说明框。锚点留在正常流中，说明框在第3节直接使用绝对定位关联锚点。本例要求浏览器支持所用锚点语法。外框预留高度只服务于本例的定位观察，不是通用浮层必须写固定高度。

普通 absolute 定位以包含块为参照；锚点函数在这一定位体系中计算坐标，不把锚点自动变成父元素或包含块。这里 .anchor-stage 的 position: relative 建立绝对定位包含块。

```html
<div class="anchor-stage">
  <p class="anchor" id="sample-anchor">这是被参照的锚点</p>
  <p class="callout">这块说明跟随锚点，并取锚点宽度。</p>
</div>
```

```css
.anchor-stage { position: relative; min-height: 14rem; padding: 1rem; border: 1px solid #657080; }
.anchor { width: 14rem; max-width: 100%; padding: 0.5rem; margin: 0; background: #d8eafa; }
.callout { max-width: 100%; padding: 0.5rem; border: 1px solid #657080; }
/* 外框建立定位包含块，锚点与说明的边框和内边距保持可见。 */
```

配套文件：[index.html](scripts/25-anchor-positioning/index.html)、[styles.css](scripts/25-anchor-positioning/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/25-anchor-positioning/index.html)

## 2 本章使用的属性

| 完整属性名 | 中文名称／含义 | 用途或对象 |
| --- | --- | --- |
| anchor-name | 锚点名称 | 在参照元素上定义名字 |
| position-anchor | 默认锚点 | 为定位元素选择参照 |
| position | 定位方式 | 使用 absolute 或 fixed |
| top | 上侧偏移 | 引用锚点上下边的坐标 |
| left | 左侧偏移 | 引用锚点左右边的坐标 |
| width | 宽度 | 根据锚点尺寸取值 |
| max-height | 最大高度 | 限制浮层高度以保留滚动 |
| position-try-fallbacks | 位置回退候选 | 溢出时尝试其他位置 |
| position-visibility | 定位可见性 | 控制条件隐藏策略 |
| overflow | 溢出处理 | 浮层长内容的内部滚动 |

anchor() 和 anchor-size() 是值函数；--sample 和 --help 是本例锚点名字，不是自定义属性变量。@position-try 是定义候选的 @ 规则，其中的同名设置属于描述符；popover、popovertarget 是 HTML 属性。

## 3 命名、关联与 anchor() 坐标

anchor-name 的名字以两个连字符开头。把 --sample 写在锚点上，再在 absolute 定位的说明框上用 position-anchor 选择它；anchor() 省略名字时使用这个默认锚点。

top: anchor(bottom) 把说明框的上边放到锚点下边的位置；calc() 再增加 0.5rem 间隔。left: anchor(left) 对齐左边。函数返回的是相应包含块坐标体系中的偏移长度，不是把“bottom”当作另一个 top 值的别名。

边与偏移属性必须同轴：top 可以引用 top、bottom，不能用 anchor(left) 代表垂直位置。函数逗号后的 4rem、1rem 是锚点无效时的数值回退；它不代表碰到视口边缘自动换位置。

锚点必须可参与相应布局，作用域与布局先后也会影响其可用性；本例使用在说明框之前、同一包含块内的正常流锚点。不要建立循环定位依赖。重复名称可能选中其他可接受锚点，本例为每组使用不同名字，避免把 class 复用误当作锚点隔离。

```html
<div class="anchor-stage">
  <p class="anchor" id="sample-anchor">这是被参照的锚点</p>
  <p class="callout">这块说明跟随锚点，并取锚点宽度。</p>
</div>
```

```css
.anchor { anchor-name: --sample; }
.callout {
  position: absolute;
  position-anchor: --sample;
  position-visibility: always;
  margin: 0;
  top: calc(anchor(bottom, 4rem) + 0.5rem);
  left: anchor(left, 1rem);
  width: anchor-size(width, 12rem);
}
/* 改变锚点宽度或上外边距：说明框跟随；临时取消anchor-name时使用函数内回退。 */
```

配套文件：[index.html](scripts/25-anchor-positioning/index.html)、[styles.css](scripts/25-anchor-positioning/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/25-anchor-positioning/index.html)

## 4 anchor-size() 与函数回退

anchor-size(width, 12rem) 取默认锚点的宽度；不能取到有效锚点尺寸时用 12rem。height 取高度；inline 与 block 按锚点书写模式取相应维度。它解决尺寸关联，anchor() 解决边坐标关联。本节专门观察函数内回退机制，因此刻意保留可选回退参数并移除锚点做对照；普通示例不必无条件添加这些参数。

函数还可用于支持范围内的尺寸、偏移与外边距属性，但本章只把它用于 width。不同版本支持范围可能不同，本例直接使用 width 中的函数语法。

在开发者工具中将锚点宽度改为 18rem，观察说明框宽度；再临时取消 anchor-name，观察 12rem 尺寸及前节坐标回退。本例显式用 position-visibility: always 避免条件隐藏干扰这次观察；此值并不阻止其他隐藏机制。

函数内回退只在浏览器已认识该函数时有意义。浏览器完全不认识 anchor-size() 时会忽略整条声明，不能把函数内回退误当作旧浏览器兼容策略。

```html
<p class="anchor" id="sample-anchor">这是被参照的锚点</p>
```

```css
.anchor { anchor-name: --sample; }
.callout {
  position: absolute;
  position-anchor: --sample;
  position-visibility: always;
  margin: 0;
  top: calc(anchor(bottom, 4rem) + 0.5rem);
  left: anchor(left, 1rem);
  width: anchor-size(width, 12rem);
}
/* 改变锚点宽度或上外边距：说明框跟随；临时取消anchor-name时使用函数内回退。 */
```

配套文件：[index.html](scripts/25-anchor-positioning/index.html)、[styles.css](scripts/25-anchor-positioning/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/25-anchor-positioning/index.html)

## 5 Popover 管开关，锚点管位置

popover="auto" 声明原生弹出内容，按钮用 popovertarget 对应其 id。默认切换显示状态；关闭按钮用 popovertargetaction="hide" 明确关闭。auto 浮层还支持轻量关闭，例如点击外部或按 Escape；这不是阻止用户操作背景的模态对话框。

打开的 Popover 进入顶层（top layer），可越过普通祖先的 overflow 裁剪和层叠上下文。锚点定位本身不会把任意元素送入顶层，也不等于增大 z-index。

Popover 与触发按钮可形成隐式锚点关联，但仍需位置 CSS。本例明确命名 --help，便于追踪参照。默认 Popover 的 inset: 0 与 margin: auto 会居中；切换锚点布局前必须处理这些默认值，否则很容易看到“有关联却没对齐”。

面板直接使用锚点布局，宽度上限和内部滚动用于不同正常视口与内容长度。Tab、Enter 和 Escape 由原生 HTML 控件处理，不需要额外鼠标事件脚本。

```html
<div class="edge-zone">
  <button id="help-trigger" type="button" popovertarget="help-panel">打开填写说明</button>
  <section id="help-panel" popover="auto" aria-labelledby="help-title">
    <h2 id="help-title">填写说明</h2>
    <p>标题应简短明确，详细说明放在正文。本面板可用关闭按钮或 Escape 退出。</p>
    <p>调整窗口，观察面板在边缘附近选择不同位置。</p>
    <button class="close" type="button" popovertarget="help-panel" popovertargetaction="hide">关闭说明</button>
  </section>
</div>
```

```css
#help-panel { padding: 1rem; border: 2px solid #657080; color: #17212b; background: #fff; }
#help-panel { overflow: auto; }
/* 长内容保留内部滚动；下一节直接设置锚点定位。 */

```

配套文件：[popover.html](scripts/25-anchor-positioning/popover.html)、[styles.css](scripts/25-anchor-positioning/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/25-anchor-positioning/popover.html)

## 6 position-try-fallbacks 处理边缘溢出

position-try-fallbacks 给出默认位置放不下时的候选。浏览器通常按列表顺序尝试，选择不溢出包含块的第一个方案。它检测的是边界溢出，不会自动避让页面里所有其他元素。

- flip-inline 沿行内方向翻转，本例水平书写时是左右换位。
- flip-block 沿块方向翻转，本例是上下换位。
- flip-block flip-inline 是同一个候选，同时翻转两轴。
- 逗号分隔不同候选；只列两个单轴候选，不能替代第三个双轴候选。

本例默认下方、左边对齐，候选包含右下角常需的双轴翻转。若所有候选都不能容纳，浏览器会退回原始位置；因此“提供回退列表”不等于“永不溢出”。浏览器还可能保留上一次可用候选，不要求一有空间就立即跳回默认位置。

```html
<button id="help-trigger" type="button" popovertarget="help-panel">打开填写说明</button>
```

```css
#help-trigger { anchor-name: --help; }
#help-panel {
  position: fixed;
  position-anchor: --help;
  position-visibility: always;
  inset: auto;
  margin: 0;
  top: calc(anchor(bottom) + 0.5rem);
  left: anchor(left);
  width: min(20rem, calc(100vw - 2rem));
  max-height: calc(100vh - 2rem);
  position-try-fallbacks: flip-inline, flip-block, flip-block flip-inline, --inside-viewport;
}
/* 默认在锚点下方且左边对齐；右下角可能需要同时翻转两个方向。 */
```

配套文件：[popover.html](scripts/25-anchor-positioning/popover.html)、[styles.css](scripts/25-anchor-positioning/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/25-anchor-positioning/popover.html)

## 7 自定义候选与长内容

@position-try --inside-viewport 定义一个命名候选，只有被 position-try-fallbacks 引用才会参与尝试。规则内的 top、left、width、max-height 等是允许的定位或尺寸描述符，不是一个任意 CSS 声明块；不要往里面写 color 或 display 来改变开关状态。

最后这个候选尝试将面板放到视口内左上角，并把宽度限制在视口减去两边各 1rem。max-height 加上普通规则中的 overflow: auto，让很长的内容仍可滚动。使用 fixed 且处于顶层的这个例子，包含块对应视口；不要把同一组视口长度机械套到局部 absolute 浮层。

position-visibility 还提供条件隐藏策略，具体值及实现仍在演进；隐藏一个必要操作可能让用户无法继续。本例选择 always，并用候选和尺寸限制保持可用，仍需检查极小视口、放大文字和滚动离开锚点的实际行为。

```html
<button class="close" type="button" popovertarget="help-panel" popovertargetaction="hide">关闭说明</button>
```

```css
@position-try --inside-viewport {
  top: 1rem;
  left: 1rem;
  right: auto;
  bottom: auto;
  width: calc(100vw - 2rem);
  max-height: calc(100vh - 2rem);
}
/* 前三个候选都放不下时，尝试视口内左上角；内容过长由原有overflow:auto滚动。 */
```

配套文件：[popover.html](scripts/25-anchor-positioning/popover.html)、[styles.css](scripts/25-anchor-positioning/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/25-anchor-positioning/popover.html)

## 8 区分位置候选、函数回退与兼容问题

函数内数值回退处理无法解析的锚点，位置候选处理合法锚点附近空间不足；浏览器是否认识语法是另一件事。本章直接使用已说明的现代环境，不再增加旧浏览器检测、静态说明或无效开关隐藏规则。

不要用旧版 position-try-options 等历史名称凑未经核对的组合。语法被接受也不能保证实现所有边界，仍要实际打开面板、移动锚点、调整宽高并检查位置。

调试时依次确认：目标是否打开且产生盒子、名称和作用域是否匹配、position 是否为 absolute/fixed、默认 inset/margin 是否已处理、坐标轴是否相符、默认位置和候选是否放得下。最后用键盘检查关闭和焦点，不能只凭截图判断交互完整。

这个小节整理检查顺序，不增加另一份重复浮层代码。


## 本章小结

- 锚点名字建立参照，anchor() 计算边坐标，anchor-size() 取得尺寸。
- 函数内的数值回退、位置候选和浏览器兼容回退解决不同问题。
- Popover 负责原生开关及顶层，CSS 负责位置；普通锚点元素不会自动进入顶层。
- 默认位置、双轴翻转、极小视口与长内容都需要独立观察。

## 练习

（1）把基础锚点宽度增大，再增加其上外边距。标准：说明框同宽并跟随位置；取消锚点名称时出现函数内给出的尺寸和坐标回退。

（2）把浮层触发按钮临时固定到视口右下角，打开浮层。标准：面板留在视口内且关闭按钮可操作；去掉双轴候选后重新比较，注意最后一个自定义候选可能接手。

（3）保持锚点关联和全部候选不变，只增加面板内容并把视口缩为360×500。标准：面板可滚动至关闭按钮，可用 Enter 打开和 Escape 关闭，记录最终使用的候选。

### 提示

改变内容后重新打开浮层，避免把浏览器保留的既有候选误解为初始选择。当前环境测试不代表旧版浏览器已经验证。

### 解析

（1）根字号16px时，18rem是288px；改变锚点宽度后两盒同宽。只取消名称后，说明框width回到192px，top为72px、left为16px（均按定位包含块计算），与锚点原位置无关。

（2）右下角通常要同时左右、上下翻转；去掉双轴候选仍可能由最后的命名候选把面板移到左上角。因此“仍在视口内”不能证明双轴翻转没有作用，要比较实际坐标。

（3）长内容会改变面板高度及候选能否放下。最大高度与内部滚动共同让关闭按钮可达；Popover原生状态负责开关，位置候选只负责布局。


## 参考与引用来源

| 来源站点 | 核查定位与对应内容 |
| --- | --- |
| W3C | [CSS Anchor Positioning Level 1 §2–3](https://www.w3.org/TR/css-anchor-position-1/#target) 的可接受锚点、默认参照和坐标函数；[§5](https://www.w3.org/TR/css-anchor-position-1/#sizing) 的锚点尺寸；[§6](https://www.w3.org/TR/css-anchor-position-1/#fallback) 的溢出候选与条件可见性。该模块仍在演进，规范新增语法与当前实现不应混为一谈。 |
| MDN | [Using anchor positioning](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Anchor_positioning/Using) 的命名、关联和隐式锚点；[anchor() 的坐标轴兼容](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Values/anchor#compatibility_of_inset_properties_and_anchor-side_values)、[anchor-size()](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Values/anchor-size#description) 的函数回退；[position-try-fallbacks](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/position-try-fallbacks#description) 的候选次序、双轴翻转和失败处理；[@position-try 的 Descriptors](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/At-rules/@position-try#descriptors)；[position-visibility](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/position-visibility#description) 的条件隐藏；[Using the Popover API](https://developer.mozilla.org/en-US/docs/Web/API/Popover_API/Using#styling_popovers) 的默认定位与键盘行为。 |
| WHATWG HTML | [The popover attribute](https://html.spec.whatwg.org/multipage/popover.html#the-popover-attribute) 的显示状态、轻量关闭与顶层关系。 |
| Python 3.12 | [http.server 命令行](https://docs.python.org/3.12/library/http.server.html#command-line-interface) 的本地服务。 |
